# Day 5: Classification — Student Practice Notebook

**Dataset:** T-ECD Retail Classroom  
**Environment:** Python 3.13.x, Jupyter Notebook, `.venv`

## Learning outcomes
- Explain classification as supervised learning.
- Identify `action_type` as the target in the approved retail event schema.
- Train and evaluate a multiclass classifier.
- Interpret accuracy, precision, recall, F1 and a confusion matrix.
- Recognise class imbalance and its effect on evaluation.

**Dataset continuity:** This follows the approved dataset journey. Day 4 uses TabReD Delivery ETA for regression; this session uses T-ECD Retail for behavioural analytics/classification. Do not substitute another dataset.

**Scope:** The exercise predicts the recorded `action_type` from attributes of the same event. It does not predict future behaviour or establish customer intent.

## 1. Dataset location and requirements

Expected folder: `data/T-ECD_Retail_Classroom/` containing the 14 daily Parquet event partitions.

Known schema: `timestamp`, `user_id`, `item_id`, `subdomain`, `action_type`, `os`.

The classroom subset is approximately 172.81 MiB. Earlier profiling found four action types, with `views` accounting for approximately 95.76% of events. Class imbalance is therefore central to this lesson.

Install if required:

```powershell
python -m pip install pandas pyarrow numpy matplotlib scikit-learn
```

**Licence:** T-ECD Retail is CC BY-NC-SA 4.0. Preserve attribution and licence information when redistributing permitted material.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
DATA_DIR = Path("data") / "T-ECD_Retail_Classroom"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Dataset folder not found: {DATA_DIR.resolve()}. "
        "Check extraction location or update DATA_DIR."
    )
print("Dataset folder:", DATA_DIR.resolve())

## 2. Locate and load the Parquet partitions

The notebook discovers Parquet files recursively rather than relying on specific partition filenames. It reads only the six documented columns.

In [ ]:
required_columns = [
    "timestamp", "user_id", "item_id",
    "subdomain", "action_type", "os"
]
parquet_files = sorted(DATA_DIR.rglob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"No Parquet files found under {DATA_DIR.resolve()}"
    )

print("Parquet partitions found:", len(parquet_files))
frames = [
    pd.read_parquet(file_path, columns=required_columns)
    for file_path in parquet_files
]
events = pd.concat(frames, ignore_index=True)

print("Combined records:", f"{len(events):,}")
display(events.head())

## 3. Validate the schema and inspect the target

Check the loaded fields, data types, missing values and the class distribution before modelling.

In [ ]:
assert set(required_columns).issubset(events.columns)

display(events.dtypes.to_frame("dtype"))
display(events[required_columns].isna().sum().to_frame("missing_count"))

action_counts = events["action_type"].value_counts(dropna=False)
display(action_counts.to_frame("event_count"))
display(
    events["action_type"].value_counts(normalize=True, dropna=False)
    .mul(100).round(2).to_frame("percentage")
)
print("Distinct action types:", events["action_type"].nunique())

## 4. Visualise class imbalance

A classifier can obtain apparently high accuracy by predicting only the dominant class. Accuracy alone is not enough when action types are unevenly represented.

In [ ]:
plt.figure(figsize=(9, 4))
action_counts.sort_values(ascending=False).plot(kind="bar")
plt.title("T-ECD Retail: Events by action type")
plt.xlabel("Action type")
plt.ylabel("Event count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

print(f"Majority-class share: {action_counts.iloc[0] / action_counts.sum() * 100:.2f}%")

## 5. Create a reproducible classroom sample

The dataset contains millions of records. To keep the exercise responsive on typical student laptops, use a reproducible sample of up to 100,000 records. This is a sample from the approved T-ECD dataset, not a replacement dataset.

In [ ]:
MAX_SAMPLE_ROWS = 100_000
model_data = events.dropna(subset=["action_type"]).copy()

if len(model_data) > MAX_SAMPLE_ROWS:
    model_data = model_data.sample(
        n=MAX_SAMPLE_ROWS, random_state=RANDOM_STATE
    )

model_data = model_data.reset_index(drop=True)
print("Modelling records:", f"{len(model_data):,}")
display(model_data["action_type"].value_counts().to_frame("event_count"))

## 6. Feature preparation

For this introductory model:
- Target: `action_type`
- Categorical predictors: `subdomain`, `os`
- Time-derived predictors: hour and day of week
- Excluded: raw `user_id` and `item_id` identifiers, to avoid high-cardinality memorisation.

The target and predictors come from the same event record; interpret the exercise accordingly.

In [ ]:
model_data["timestamp"] = pd.to_datetime(
    model_data["timestamp"], errors="coerce"
)
model_data["event_hour"] = model_data["timestamp"].dt.hour
model_data["event_day_of_week"] = model_data["timestamp"].dt.dayofweek

feature_columns = [
    "subdomain", "os", "event_hour", "event_day_of_week"
]
target_column = "action_type"

model_data = model_data.dropna(
    subset=feature_columns + [target_column]
).copy()

X = model_data[feature_columns]
y = model_data[target_column].astype(str)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", sorted(y.unique().tolist()))
display(X.head())

## 7. Create stratified training and test partitions

Stratification approximately preserves each class's proportion in both partitions. Keep the test set for final evaluation; do not repeatedly tune against it.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE,
    stratify=y
)

print("Training records:", f"{len(X_train):,}")
print("Test records:", f"{len(X_test):,}")
display(y_train.value_counts(normalize=True).mul(100).round(2).to_frame("train_pct"))
display(y_test.value_counts(normalize=True).mul(100).round(2).to_frame("test_pct"))

## 8. Build preprocessing steps

Categorical fields are imputed and one-hot encoded. Numeric fields are median-imputed and standardised. The preprocessing is part of the pipeline, so it is fitted on training data only.

In [ ]:
categorical_features = ["subdomain", "os"]
numeric_features = ["event_hour", "event_day_of_week"]

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(transformers=[
    ("categorical", categorical_pipeline, categorical_features),
    ("numeric", numeric_pipeline, numeric_features),
])
print("Preprocessing pipeline configured.")

## 9. Establish a majority-class baseline

A simple baseline always predicts the most frequent class. A more complex classifier should demonstrate value beyond this reference point.

In [ ]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent")),
])
baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)

print("Baseline accuracy:",
      round(accuracy_score(y_test, baseline_predictions), 4))
print(classification_report(
    y_test, baseline_predictions, zero_division=0
))

## 10. Train a Logistic Regression classifier

Despite its name, Logistic Regression is a classification algorithm. `class_weight="balanced"` adjusts class weights inversely to class frequencies in the training data. This is an experiment, not a guarantee of improved business performance.

In [ ]:
classification_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])
classification_model.fit(X_train, y_train)
predictions = classification_model.predict(X_test)
print("Predictions generated:", len(predictions))

## 11. Evaluate the classifier

- **Accuracy:** fraction of all predictions that are correct.
- **Precision:** of the records predicted as a class, how many truly belong to it?
- **Recall:** of the actual records in a class, how many were identified?
- **F1-score:** harmonic mean of precision and recall.
- **Macro average:** each class receives equal weight.
- **Weighted average:** each class is weighted by its support.

For this imbalanced multiclass task, examine macro F1 and per-class results, not accuracy alone.

In [ ]:
metrics_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Macro Precision", "Macro Recall",
               "Macro F1", "Weighted F1"],
    "Score": [
        accuracy_score(y_test, predictions),
        precision_score(y_test, predictions, average="macro", zero_division=0),
        recall_score(y_test, predictions, average="macro", zero_division=0),
        f1_score(y_test, predictions, average="macro", zero_division=0),
        f1_score(y_test, predictions, average="weighted", zero_division=0),
    ],
})
display(metrics_summary)
print(classification_report(y_test, predictions, zero_division=0))

## 12. Compare with the baseline

Compare both accuracy and macro F1. Did Logistic Regression improve over the majority-class baseline? Which classes remain difficult to identify?

In [ ]:
comparison = pd.DataFrame({
    "Majority baseline": {
        "Accuracy": accuracy_score(y_test, baseline_predictions),
        "Macro F1": f1_score(
            y_test, baseline_predictions, average="macro", zero_division=0
        ),
    },
    "Logistic Regression": {
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro F1": f1_score(
            y_test, predictions, average="macro", zero_division=0
        ),
    },
}).T
display(comparison)

## 13. Interpret the confusion matrix

Rows represent actual classes; columns represent predicted classes. Diagonal values are correct predictions. Off-diagonal values show class confusions.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, predictions,
    labels=classification_model.named_steps["classifier"].classes_,
    xticks_rotation=45, cmap="Blues",
    colorbar=False, ax=ax,
)
ax.set_title("T-ECD Retail — Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()

## 14. Business interpretation

Imagine a retail analytics team wants to understand recorded event activity. This classifier demonstrates how selected event attributes relate to the recorded `action_type`.

Before considering operational use, ask:
1. Is it meaningfully better than the majority-class baseline?
2. Does it identify less frequent action types?
3. Which misclassifications matter most?
4. Would it generalise to later dates or new users?
5. Would all predictors be available at the time a prediction is required?

**Limitation:** This is classification of an event's recorded label using attributes from that same event. It is not future-event forecasting and does not establish customer intent or causation.

## 15. Hands-on challenge

**Task A — Class distribution**
- Identify the most and least frequent action types.
- Explain why accuracy alone may mislead.

**Task B — Model comparison**
- Compare the majority baseline and Logistic Regression.
- Focus on macro F1 and per-class recall.
- Identify one poorly predicted class.

**Task C — Feature experiment**
- Train another model using only `subdomain` and `os`.
- Compare macro F1 with the model that also uses time-derived features.
- Explain whether time features appear useful in this sample.

**Task D — Business communication**
Write a short manager summary stating what is predicted, whether the model improves on baseline, one class-level weakness and one limitation before operational use.

Do not claim that this model predicts future customer behaviour.

## 16. Knowledge check

1. What is the difference between regression and classification?
2. Why can a majority-class classifier have high accuracy on imbalanced data?
3. What does recall measure?
4. Why is macro F1 useful when classes are imbalanced?
5. What does a confusion matrix show?

## Submission checklist
- [ ] Notebook runs from top to bottom.
- [ ] Approved T-ECD Parquet partitions are loaded.
- [ ] Target distribution is inspected.
- [ ] Reproducible sample is created.
- [ ] Raw user/item identifiers are excluded.
- [ ] Train/test split is stratified.
- [ ] Majority baseline and Logistic Regression are evaluated.
- [ ] Accuracy, precision, recall, F1 and confusion matrix are reviewed.
- [ ] Business interpretation acknowledges imbalance and limitations.

**Next session:** Continue with the next classification algorithm in the approved course sequence.